# 02 — Boundary Conditions, Optical Depth, and PINN Failure

A PINN that works for one parameter regime may fail when the physical solution develops a shorter spatial scale. This notebook separates three questions:

1. How should a boundary condition be enforced?
2. Does the collocation set resolve the smallest important scale?
3. Which diagnostics reveal a localized failure?

Everything here is written in the nondimensional formulation you derived in Notebook 00: the coordinate is $\hat{z}\in[0,1]$, the source level is `SOURCE`, the boundary value is `u_0`, and the dimensionless group multiplying $(u-\texttt{SOURCE})$ is the optical depth $\tau$.

## Learning goals

- reuse a derivation across notebooks instead of re-entering it;
- explain what a hard boundary constraint is, why it is used, and what it costs;
- compare soft and hard boundary enforcement on the same problem;
- identify the characteristic transport scale $1/\tau$;
- diagnose errors hidden by global metrics;
- implement boundary-biased collocation sampling;
- distinguish constraint, sampling, representation, and optimization failures.

Expected working time: about 3–3.5 hours.

## 1. Inherit your Notebook 00 solutions

This notebook does not ask you to paste anything. It runs your completed `00_setting_up_the_problem.ipynb` and picks up the definitions you already wrote there, so a change to your derivation propagates here automatically instead of drifting out of sync with a stale copy.

The cell below executes Notebook 00 in this kernel and quiets its output. Everything it defines becomes available here:

| Inherited from Notebook 00 | What it is |
| --- | --- |
| `SOURCE` | nondimensional source level, fixed to 1 by your intensity normalization |
| `u_0` | nondimensional inflow, the boundary value $u(\hat{z}=0)$ |
| `OPACITY` | the optical depth $\tau$ you chose for the baseline problem |
| `coordinate_derivative` | $du/d\hat{z}$ by automatic differentiation |
| `transport_residual` | your nondimensional residual $r(\hat{z})$ |
| `exact_solution_torch` | the supplied analytic solution |

If the load fails, the traceback comes from Notebook 00 itself: finish that notebook first, save it, then rerun this cell. Instructors running the worked version can point the filename at `00_setting_up_the_problem_instructor.ipynb`.

In [ ]:
from IPython.utils.capture import capture_output

with capture_output():
    get_ipython().run_line_magic("run", "00_setting_up_the_problem.ipynb")

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn

torch.set_default_dtype(torch.float32)
torch.set_num_threads(1)

print("inherited from Notebook 00:")
print(f"  SOURCE  = {SOURCE}")
print(f"  u_0     = {u_0}")
print(f"  OPACITY = {OPACITY}   (your baseline optical depth tau)")
print(f"  residual check on the analytic solution: "
      f"{float(transport_residual(x, exact_solution_torch(x, OPACITY, SOURCE, u_0), OPACITY, SOURCE).detach().abs().max()):.2e}")

### What this notebook adds

Notebook 00 gave you the physics: a residual, a boundary value, and an analytic solution to check against. It did not give you a PINN. The next cell supplies the machinery this notebook needs on top of that — a network, a sampler, the two loss terms, an evaluation routine, and plotting helpers.

None of it is new physics, and none of it is an exercise; read it once so you know what the experiment driver is doing, then move on. The one thing worth noticing is `model_residual`: it does not reimplement the residual, it wraps **your** `transport_residual` from Notebook 00 so that the equation being solved here is exactly the one you derived.

In [ ]:
import random

import numpy as np


def set_seed(seed: int) -> None:
    """Seed Python, NumPy, and PyTorch together."""
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))


def make_generator(seed: int) -> torch.Generator:
    """An independent RNG, so sampling does not disturb weight initialization."""
    return torch.Generator(device="cpu").manual_seed(int(seed))


def dense_grid(n_points: int = 1001) -> torch.Tensor:
    """Evenly spaced column tensor on [0, 1], for evaluation only."""
    return torch.linspace(0.0, 1.0, int(n_points)).reshape(-1, 1)


class MLP(nn.Module):
    """Fully connected tanh network mapping one coordinate to one intensity."""

    def __init__(self, hidden_dim: int = 24, hidden_layers: int = 3) -> None:
        super().__init__()
        layers = [nn.Linear(1, hidden_dim), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.Tanh()]
        layers += [nn.Linear(hidden_dim, 1)]
        self.network = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


def sample_uniform_collocation(n_points: int, seed: int) -> torch.Tensor:
    """Reproducible uniform collocation points on [0, 1]."""
    return torch.rand((int(n_points), 1), generator=make_generator(seed))


def model_residual(model: nn.Module, x: torch.Tensor, opacity: float,
                   source: float = None) -> torch.Tensor:
    """Evaluate YOUR Notebook 00 residual on a model's output.

    The coordinates are cloned into a fresh differentiable leaf so that each call
    builds its own graph instead of reusing one from a previous optimizer step.
    """
    source = SOURCE if source is None else source
    x_local = x.detach().clone().requires_grad_(True)
    u = model(x_local)
    return transport_residual(x_local, u, opacity, source)


def compute_loss_terms(model: nn.Module, x_collocation: torch.Tensor, *,
                       opacity: float, source: float, u_in: float) -> dict:
    """Physics and boundary losses, kept separate so weighting stays explicit."""
    residual = model_residual(model, x_collocation, opacity, source)
    physics_loss = residual.square().mean()
    boundary_loss = (model(torch.zeros((1, 1))) - u_in).square().mean()
    return {"physics": physics_loss, "boundary": boundary_loss}


def solution_metrics(prediction, reference, residual, boundary_value, u_in) -> dict:
    error = prediction - reference
    return {
        "relative_l2": float(error.norm() / reference.norm()),
        "max_abs_error": float(error.abs().max()),
        "rms_residual": float(residual.square().mean().sqrt()),
        "boundary_error": float(abs(float(boundary_value) - float(u_in))),
    }


def evaluate_model(model: nn.Module, *, opacity: float, source: float = None,
                   u_in: float = None, n_evaluation: int = 1001) -> dict:
    """Evaluate on a dense grid that was never used for collocation."""
    source = SOURCE if source is None else source
    u_in = u_0 if u_in is None else u_in
    x_eval = dense_grid(n_evaluation)
    prediction = model(x_eval).detach()
    reference = exact_solution_torch(x_eval, opacity, source, u_in)
    residual = model_residual(model, x_eval, opacity, source).detach()
    metrics = solution_metrics(prediction, reference, residual,
                               model(torch.zeros((1, 1))).detach(), u_in)
    return {"x": x_eval, "prediction": prediction, "reference": reference,
            "residual": residual, "metrics": metrics}


def count_points_in_layer(x: torch.Tensor, opacity: float) -> int:
    """Collocation points inside the characteristic width z_hat < 1/tau."""
    return int(torch.sum(x.detach() < 1.0 / float(opacity)))


def boundary_offset(model: nn.Module, u_in: float) -> float:
    """|u(0) - u_in| for a model, reported rather than asserted."""
    return float((model(torch.zeros((8, 1))).detach() - u_in).abs().max())


def plot_solution_comparison(x, reference, prediction, title="", xlim=None):
    fig, ax = plt.subplots(figsize=(6.4, 4.0))
    ax.plot(x.detach(), reference.detach(), label="exact")
    ax.plot(x.detach(), prediction.detach(), "--", label="PINN")
    if xlim is not None:
        ax.set_xlim(*xlim)
    ax.set_xlabel(r"$\hat{z}$")
    ax.set_ylabel(r"$u(\hat{z})$")
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.25)
    return fig, ax


def plot_loss_history(history, title=""):
    fig, ax = plt.subplots(figsize=(6.4, 4.0))
    for name, values in history.items():
        if values:
            ax.semilogy(values, label=name)
    ax.set_xlabel("recorded step")
    ax.set_ylabel("loss")
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.25)
    return fig, ax


def plot_error_and_residual(x, absolute_error, residual, title=""):
    fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.0))
    axes[0].semilogy(x.detach(), absolute_error.detach() + 1e-16)
    axes[0].set_xlabel(r"$\hat{z}$")
    axes[0].set_ylabel("|error|")
    axes[0].set_title("pointwise error")
    axes[1].plot(x.detach(), residual.detach())
    axes[1].set_xlabel(r"$\hat{z}$")
    axes[1].set_ylabel(r"$r(\hat{z})$")
    axes[1].set_title("residual on the evaluation grid")
    for ax in axes:
        ax.grid(alpha=0.25)
    fig.suptitle(title)
    fig.tight_layout()
    return fig, axes


def plot_collocation(x_collocation, title=""):
    fig, ax = plt.subplots(figsize=(6.4, 1.8))
    ax.plot(x_collocation.detach(), torch.zeros_like(x_collocation.detach()), "|",
            markersize=14)
    ax.set_yticks([])
    ax.set_xlim(-0.02, 1.02)
    ax.set_xlabel(r"$\hat{z}$")
    ax.set_title(title)
    return fig, ax


print("supplied components ready")

## 2. Enforce the inflow condition exactly

### The problem a boundary condition is solving

Notebook 00 ended on a specific observation. Every function in the family

$$
u_C(\hat{z}) = \texttt{SOURCE} + C\,e^{-\tau\hat{z}}
$$

drives the residual to zero, for any constant $C$. The differential equation alone does not identify a solution; it identifies a one-parameter family of them. The boundary value $u(0)=u_0$ is what selects the single physical member, $C = u_0 - \texttt{SOURCE}$.

So a PINN has to be told about the boundary somehow. There are two ways to do it, and they differ in whether the wrong members of the family are *penalized* or *unrepresentable*.

### Soft enforcement, and what it costs

The soft approach adds a penalty term to the objective:

$$
\mathcal{L} = \lambda_f\,\mathcal{L}_f + \lambda_b\,\mathcal{L}_b,
\qquad
\mathcal{L}_b = \left[u_\theta(0) - u_0\right]^2 .
$$

This works, and it is the default in much of the PINN literature because it generalizes to essentially any constraint. But it has real costs:

- **It introduces a hyperparameter with no principled value.** $\lambda_b$ trades the equation against the boundary. Too small and you recover the failure from Notebook 00, where the network settles on the wrong member of the family. Too large and the boundary term dominates the gradient, the physics is effectively down-weighted, and the optimization becomes ill-conditioned. There is no universally correct setting; it depends on $\tau$, the architecture, and the optimizer.
- **The constraint is only ever approximate.** $u_\theta(0)$ approaches $u_0$; it does not equal it. A residual boundary error of $10^{-3}$ is a real physical error in the answer, not a numerical detail.
- **The constraint is violated throughout training.** The optimizer's whole trajectory passes through models that do not satisfy the inflow condition, which is wasted effort on physically inadmissible states.
- **It muddies the diagnostics.** A single total loss now mixes two quantities that mean different things. A small $\mathcal{L}$ could be a well-solved equation with a bad boundary, or the reverse.

### Hard enforcement: build the condition into the model

The hard approach changes the model instead of the objective. Write the prediction as

$$
u_\theta(\hat{z}) = u_0 + g(\hat{z})\,N_\theta(\hat{z}),
\qquad g(0) = 0 ,
$$

where $N_\theta$ is the unconstrained network. Evaluate at the boundary and the second term vanishes identically, so $u_\theta(0) = u_0$ for **every** parameter vector $\theta$ — at initialization, at every intermediate step, and at convergence. The condition is not learned, approached, or weighted. It is a property of the function space the network is searching in.

Note what the network's job becomes. $N_\theta$ no longer represents the solution; it represents the *deviation* from the boundary value, and $g$ says how that deviation is allowed to switch on as you move away from the wall. The simplest choice that works on $[0,1]$ is $g(\hat{z}) = \hat{z}$.

This construction is older than PINNs — it is the trial-function idea from Lagaris, Likas and Fotiadis (1998), which built exact boundary satisfaction into a neural solution ansatz long before the physics-informed framing became standard.

### Why it is worth using here

1. **One fewer hyperparameter.** $\lambda_b$ disappears entirely, and with it the tuning problem it created.
2. **Exactness.** The boundary error is zero to machine precision instead of small.
3. **A single-objective optimization.** With the boundary built in, the loss contains only the physics term. The two gradients can no longer conflict or differ by orders of magnitude in scale.
4. **The residual becomes trustworthy evidence.** This is the important one. Notebook 00 asked why a low residual is insufficient proof that a PINN solved the problem; the answer was the one-parameter family. A hard constraint collapses that family to a single member, because no admissible model can take any other value at $\hat{z}=0$. Once the boundary holds by construction, a small residual *does* mean the problem is solved.

### Choosing $g$, and where the approach runs out

$g$ must vanish at the boundary and stay nonzero in the interior — if $g$ were zero anywhere inside the domain, it would pin the solution to $u_0$ there and remove the network's freedom. Beyond that, the choice is an inductive bias: $g(\hat{z}) = \hat{z}$ makes the correction grow linearly away from the wall, and a $g$ that stays very small across most of the domain would force $N_\theta$ to produce very large outputs to compensate, which is a conditioning problem.

The method is not free. It is easy here because the domain is an interval with one Dirichlet condition at one end. It becomes harder when:

- the geometry is complicated, so a smooth function vanishing on the whole boundary is itself a construction problem (approximate distance functions and R-functions exist for this);
- conditions are Neumann or Robin, constraining a derivative rather than a value, which no simple multiplicative ansatz enforces;
- several boundaries carry different conditions, requiring a $g$ that vanishes appropriately on each.

That is why soft penalties remain widespread despite hard constraints being the better tool when they are available. For this problem, they are available.

### What a hard constraint does *not* buy you

It fixes the boundary and nothing else. A model can satisfy $u(0)=u_0$ exactly and still be badly wrong across the interior. Sections 5 through 7 are built on exactly that gap: the constraint is perfect throughout, and the solution still fails.

Implement the transformation below. Choose the simplest useful $g(\hat{z})$ for the interval $[0,1]$.

In [ ]:
class HardInflowModel(nn.Module):
    """Wrap a network so that u(0) = u_in holds identically.

    u_in is the NONDIMENSIONAL boundary value u_0 from Notebook 00, not the
    physical intensity INFLOW.
    """

    def __init__(self, core_model: nn.Module, u_in: float) -> None:
        super().__init__()
        self.core_model = core_model
        self.u_in = float(u_in)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        raw_output = self.core_model(x)

        # TODO: transform raw_output so that u(0)=u_in identically.
        # Use the form u = u_in + g(x) * raw_output with g(0) = 0.
        constrained_output = None

        return constrained_output


set_seed(21)
hard_test_model = HardInflowModel(MLP(hidden_dim=8), u_in=u_0)

print(f"|u(0) - u_0| before any training: {boundary_offset(hard_test_model, u_0):.3e}")
print("(this should be exactly 0.0, not merely small)")

<details>
<summary><strong>Conceptual hint</strong></summary>

Multiply the unconstrained network output by a function that vanishes at the boundary. The transformation should leave the network free to alter the solution away from $\hat{z}=0$, so that function must not vanish anywhere else in $[0,1]$.

</details>

<details>
<summary><strong>PyTorch hint</strong></summary>

The choice $g(\hat{z})=\hat{z}$ gives `self.u_in + x * raw_output`.

</details>

## 3. Train a supplied model on a supplied collocation set

Adapt the training loop so that it accepts an already-created model and an already-created set of coordinates. This separation lets the experiment driver in later sections change the constraint and the sampling strategy without rewriting the optimizer logic.

Both loss terms are computed by `compute_loss_terms`, which is built on your Notebook 00 residual. Weighting them is your job, and it is deliberately left outside that function: the weights are part of the experimental design, not part of the physics.

In [1]:
def train_given_model(
    model: nn.Module,
    x_collocation: torch.Tensor,
    *,
    opacity: float,
    source: float,
    u_in: float,
    steps: int,
    learning_rate: float,
    lambda_physics: float,
    lambda_boundary: float,
    record_every: int = 10,
) -> dict[str, list[float]]:
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history = {"total": [], "physics": [], "boundary": []}

    for step in range(steps):
        # TODO: clear stored parameter gradients.

        # TODO: compute the two loss terms with compute_loss_terms.
        terms = None

        # TODO: form the weighted scalar total loss.
        total_loss = None

        # TODO: backpropagate and update the parameters.

        if step % record_every == 0 or step == steps - 1:
            # TODO: append detached Python floats for all three histories.
            pass

    return history

<details>
<summary><strong>Conceptual hint</strong></summary>

A complete update is: clear gradients, evaluate the scalar loss, differentiate it, step the optimizer. The history is for logging only and must never retain a computational graph, or memory will grow with every step.

</details>

<details>
<summary><strong>PyTorch hint</strong></summary>

Use `optimizer.zero_grad()`, `compute_loss_terms(model, x_collocation, opacity=opacity, source=source, u_in=u_in)`, `lambda_physics * terms["physics"] + lambda_boundary * terms["boundary"]`, `total_loss.backward()`, `optimizer.step()`, and `float(tensor.detach())`.

</details>

## 4. Soft versus hard enforcement

Compare the two formulations at $\tau = 10$, which is inside the optical-depth band Notebook 00 quoted for the troposphere but well above your baseline. Use the same architecture, collocation coordinates, number of steps, and initialization seed, so the constraint is the only thing that differs. The soft model carries a boundary penalty; the hard model does not need one and is trained with `lambda_boundary=0`.

In [ ]:
COMPARISON_TAU = 10.0
COMPARISON_STEPS = 900
COMPARISON_POINTS = 96
COMPARISON_SEED = 5

x_shared = sample_uniform_collocation(COMPARISON_POINTS, COMPARISON_SEED + 100)

set_seed(COMPARISON_SEED)
soft_model = MLP()
soft_history = train_given_model(
    soft_model,
    x_shared,
    opacity=COMPARISON_TAU,
    source=SOURCE,
    u_in=u_0,
    steps=COMPARISON_STEPS,
    learning_rate=3.0e-3,
    lambda_physics=1.0,
    lambda_boundary=10.0,
)

set_seed(COMPARISON_SEED)
hard_model = HardInflowModel(MLP(), u_0)
hard_history = train_given_model(
    hard_model,
    x_shared,
    opacity=COMPARISON_TAU,
    source=SOURCE,
    u_in=u_0,
    steps=COMPARISON_STEPS,
    learning_rate=3.0e-3,
    lambda_physics=1.0,
    lambda_boundary=0.0,
)

soft_eval = evaluate_model(soft_model, opacity=COMPARISON_TAU)
hard_eval = evaluate_model(hard_model, opacity=COMPARISON_TAU)

pd.DataFrame(
    [
        {"method": "soft", **soft_eval["metrics"]},
        {"method": "hard", **hard_eval["metrics"]},
    ]
).set_index("method")

In [ ]:
for label, evaluation, history in [
    ("soft", soft_eval, soft_history),
    ("hard", hard_eval, hard_history),
]:
    plot_solution_comparison(
        evaluation["x"],
        evaluation["reference"],
        evaluation["prediction"],
        title=f"{label.capitalize()} inflow constraint",
    )
    plt.show()
    plot_loss_history(history, title=f"{label.capitalize()}-constraint training")
    plt.show()

### Interpretation checkpoint

1. Which formulation satisfies the boundary most accurately, and why was that true before training began?
2. Which formulation requires a tunable boundary weight, and what would you have to do to choose it defensibly?
3. Does exact boundary satisfaction guarantee a small interior error? What does the table show?
4. In the hard run the boundary history is identically zero. Is that a reason to stop reporting it?
5. Why might a hard transformation become inconvenient for complicated geometries or mixed conditions?

## 5. Hold the PINN fixed and increase optical depth

For the analytic solution

$$
u(\hat{z}) = \texttt{SOURCE} + (u_0 - \texttt{SOURCE})\,e^{-\tau\hat{z}},
$$

most of the change happens over

$$
\Delta\hat{z}\sim\frac{1}{\tau}.
$$

At $\tau=1$ that scale is the whole domain. At $\tau=50$ it is the leftmost two percent of it — and notice that $\tau=50$ is deliberately outside the $0.8<\tau<20$ band Notebook 00 derived for the troposphere. The point is to push the method past the regime the problem was set up for and watch how it fails.

Use a hard boundary condition throughout, so the sweep isolates the effect of the shrinking spatial scale rather than mixing it with a constraint error. Keep the network, optimizer, number of points, and training steps fixed.

In [ ]:
def run_forward_experiment(
    *,
    opacity: float,
    sampling_function,
    n_collocation: int = 96,
    steps: int = 1200,
    seed: int = 1,
) -> dict[str, object]:
    """Create, train, and evaluate one hard-boundary PINN experiment."""
    # TODO: set the global seed before constructing the network.
    # TODO: create HardInflowModel(MLP(), u_0).
    # TODO: obtain collocation points by calling sampling_function(n_collocation, seed + 100).
    # TODO: train with train_given_model using lambda_physics=1.0 and lambda_boundary=0.0.
    # TODO: evaluate on the dense independent grid with evaluate_model.
    # TODO: return a dict with keys "model", "x_collocation", "history", "evaluation".
    return None

<details>
<summary><strong>Conceptual hint</strong></summary>

The sampler should be a callable that accepts `(n_points, seed)` and returns a column tensor. Keeping sampling choices outside the experiment driver is what lets section 7 swap in a different distribution without touching this function.

</details>

<details>
<summary><strong>PyTorch hint</strong></summary>

Call `set_seed(seed)`, construct the hard model, call `sampling_function(n_collocation, seed + 100)`, then `train_given_model(..., source=SOURCE, u_in=u_0, learning_rate=3.0e-3, lambda_physics=1.0, lambda_boundary=0.0)` and `evaluate_model(model, opacity=opacity)`.

</details>

In [ ]:
optical_depths = [0.1, 1.0, 10.0, 50.0]
sweep_results = {}
sweep_rows = []

for tau in optical_depths:
    result = run_forward_experiment(
        opacity=tau,
        sampling_function=sample_uniform_collocation,
        n_collocation=96,
        steps=1200,
        seed=1,
    )
    sweep_results[tau] = result
    sweep_rows.append(
        {
            "tau": tau,
            "layer_width": 1.0 / tau,
            "points_in_layer": count_points_in_layer(result["x_collocation"], tau),
            **result["evaluation"]["metrics"],
        }
    )

sweep_table = pd.DataFrame(sweep_rows).set_index("tau")
sweep_table

## 6. Diagnose the optically thick case locally

A global norm averages error over the entire interval. At $\tau=50$ the difficult region occupies about two percent of the domain, so a global metric can look acceptable while the physically interesting part of the solution is wrong. Inspect it directly.

In [ ]:
thick_uniform = sweep_results[50.0]
thick_eval = thick_uniform["evaluation"]
x_eval = thick_eval["x"]
absolute_error = torch.abs(thick_eval["prediction"] - thick_eval["reference"])

plot_solution_comparison(
    x_eval, thick_eval["reference"], thick_eval["prediction"],
    title="tau = 50: full domain",
)
plt.show()

plot_solution_comparison(
    x_eval, thick_eval["reference"], thick_eval["prediction"],
    title="tau = 50: magnified inflow region", xlim=(0.0, 0.12),
)
plt.show()

plot_error_and_residual(
    x_eval, absolute_error, thick_eval["residual"],
    title="tau = 50: dense-grid diagnostics",
)
plt.show()

plot_collocation(thick_uniform["x_collocation"], title="uniform collocation for tau = 50")
plt.show()

### Diagnose the failure

1. How many points lie inside one characteristic width $\hat{z} < 1/\tau$?
2. Does the global relative error communicate *where* the error occurs?
3. Is the residual small only at the training coordinates, or also between them?
4. What evidence points to a sampling problem?
5. What evidence would instead point to a boundary-constraint problem — and can that be the explanation here, given how the model was built?

## 7. Required repair: boundary-biased sampling

Generate $\zeta$ uniformly on $[0,1]$ and map it to

$$
\hat{z} = \zeta^{\,p},\qquad p>1 .
$$

This leaves the physical domain unchanged — the points still cover $[0,1]$ — but allocates more of them near the inflow boundary, which is where the solution varies fastest at large $\tau$.

In [ ]:
def sample_boundary_biased(
    n_points: int,
    seed: int,
    *,
    power: float = 3.0,
) -> torch.Tensor:
    """Return points concentrated near z_hat=0; power=1 gives uniform sampling."""
    if n_points < 1:
        raise ValueError("n_points must be positive")
    if power < 1.0:
        raise ValueError("power must be at least one")

    # TODO: sample zeta uniformly with an independent seeded generator.
    # TODO: transform zeta to z_hat = zeta**power and return it.
    return None


biased_test = sample_boundary_biased(5000, seed=33, power=3.0)
uniform_test = sample_uniform_collocation(5000, seed=33)

print(f"shapes match      : {biased_test.shape == uniform_test.shape}")
print(f"mean uniform z_hat: {float(uniform_test.mean()):.4f}")
print(f"mean biased  z_hat: {float(biased_test.mean()):.4f}  (should be markedly smaller)")
print(f"fraction of biased points below 0.02: {float((biased_test < 0.02).float().mean()):.3f}")

<details>
<summary><strong>Conceptual hint</strong></summary>

For numbers between zero and one, raising to a power greater than one makes them smaller, moving them toward the inflow boundary.

</details>

<details>
<summary><strong>PyTorch hint</strong></summary>

Create `zeta = torch.rand((n_points, 1), generator=make_generator(seed))` and return `zeta.pow(power)`.

</details>

In [ ]:
def biased_sampler(n_points: int, seed: int) -> torch.Tensor:
    return sample_boundary_biased(n_points, seed, power=3.0)


thick_biased = run_forward_experiment(
    opacity=50.0,
    sampling_function=biased_sampler,
    n_collocation=96,
    steps=1200,
    seed=1,
)

repair_table = pd.DataFrame(
    [
        {
            "sampling": "uniform",
            "points_in_layer": count_points_in_layer(thick_uniform["x_collocation"], 50.0),
            **thick_uniform["evaluation"]["metrics"],
        },
        {
            "sampling": "boundary-biased",
            "points_in_layer": count_points_in_layer(thick_biased["x_collocation"], 50.0),
            **thick_biased["evaluation"]["metrics"],
        },
    ]
).set_index("sampling")
repair_table

In [ ]:
for label, result in [("uniform", thick_uniform), ("boundary-biased", thick_biased)]:
    evaluation = result["evaluation"]
    plot_solution_comparison(
        evaluation["x"], evaluation["reference"], evaluation["prediction"],
        title=f"tau = 50 with {label} sampling", xlim=(0.0, 0.12),
    )
    plt.show()
    plot_collocation(result["x_collocation"], title=f"{label} collocation")
    plt.show()

## 8. One controlled exploration

Keep everything fixed and try exactly one additional value of the power $p$. State a prediction first: will it improve the full-domain error, the near-boundary error, both, or neither? Explain the result using the distribution of collocation coordinates.

Avoid testing many powers without a hypothesis. The objective is a controlled scientific comparison, not an unstructured search.

In [ ]:
CHOSEN_POWER = None  # TODO: choose one value other than 1 or 3.

# TODO: define a sampler using CHOSEN_POWER, run one experiment with
# run_forward_experiment at opacity=50.0, and compare it quantitatively with the
# uniform and power=3 cases in a single table.

## 9. Failure-classification checkpoint

For each statement below, label the most direct issue as **constraint**, **sampling**, **representation**, **optimization**, or **diagnostic** failure. Some cases may involve more than one category; identify the primary one and justify it.

1. The boundary value is wrong because its loss weight is zero.
2. The collocation set contains almost no points in a narrow transition region.
3. Training loss stalls at a large value for every tested seed.
4. The network cannot reproduce a sharp feature even when densely sampled and carefully optimized.
5. A global norm is small although a scientifically important local error is large.

Finish with a paragraph explaining why "increase the network size" is not an adequate default response to every failed PINN. Refer to the sweep in section 5: which of the five categories would a larger network actually have addressed there?